In [ ]:
from huggingface_hub import login

# Paste your HF token here — get from https://huggingface.co/settings/tokens
# Make sure you've accepted the LLaMA 3 license at:
# https://huggingface.co/meta-llama/Meta-Llama-3-8B
HF_TOKEN = "HUGGING_FACE_ACCESS_TOKEN"  # <-- REPLACE THIS

login(token=HF_TOKEN)
print('✅ Logged in')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\himan\.cache\huggingface\token
Login successful
✅ Logged in


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

import torch
import asyncio
import edge_tts
from playsound import playsound
import os
import uuid
import nest_asyncio

# =========================================================
# FIX NOTEBOOK ASYNC ISSUE
# =========================================================

nest_asyncio.apply()

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n🔥 Using device: {device}")

# =========================================================
# LOAD TINYLLAMA (YOUR PERSONALITY MODEL)
# =========================================================

tiny_model_path = "./final_tiny_model"

tiny_tokenizer = AutoTokenizer.from_pretrained(
    tiny_model_path
)

tiny_model = AutoModelForCausalLM.from_pretrained(
    tiny_model_path,
    torch_dtype=torch.float16
).to(device)

tiny_model.eval()

print("✅ TinyLlama loaded")

# =========================================================
# LOAD LLAMA 3 8B
# =========================================================

llama_model_name = "Qwen/Qwen2.5-7B-Instruct"

llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name
)

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to(device)

llama_model.eval()

print("✅ Llama 3 loaded")

# =========================================================
# TEXT TO SPEECH
# =========================================================

async def speak_text(text):

    filename = f"voice_{uuid.uuid4()}.mp3"

    communicate = edge_tts.Communicate(
        text=text,
        voice="en-US-AriaNeural",
        rate="+8%",
        pitch="+2Hz"
    )

    await communicate.save(filename)

    playsound(filename)

    os.remove(filename)

# =========================================================
# COMPLEX QUERY DETECTOR
# =========================================================

def is_complex_query(query):

    complex_keywords = [
        "code",
        "python",
        "explain",
        "debug",
        "compare",
        "analyze",
        "algorithm",
        "ai",
        "machine learning",
        "deep learning",
        "math",
        "write program",
        "generate code"
    ]

    query_lower = query.lower()

    if len(query.split()) > 12:
        return True

    for word in complex_keywords:
        if word in query_lower:
            return True

    return False

# =========================================================
# LLAMA 3 RESPONSE
# =========================================================

def ask_llama3(query):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a smart AI assistant. "
                "Answer accurately and clearly."
            )
        },
        {
            "role": "user",
            "content": query
        }
    ]

    input_ids = llama_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        outputs = llama_model.generate(
            input_ids,
            max_new_tokens=250,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = llama_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

# =========================================================
# TINYLLAMA PERSONALITY REWRITER
# =========================================================

def rewrite_with_tinyllama(text):

    prompt = f"""
### System:
You are a respectful AI assistant.
You ALWAYS address the user as Sir Himanshu.
Rewrite naturally and conversationally.

### Instruction:
Rewrite this answer:

{text}

### Response:
Sir Himanshu,
"""

    inputs = tiny_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        outputs = tiny_model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.65,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            eos_token_id=tiny_tokenizer.eos_token_id,
            pad_token_id=tiny_tokenizer.eos_token_id
        )

    response = tiny_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split("### Response:")[-1]

    stop_tokens = [
        "### Instruction:",
        "<|user|>",
        "<|assistant|>",
        "Q:",
        "User:"
    ]

    for token in stop_tokens:
        response = response.split(token)[0]

    response = response.strip()

    if not response.startswith("Sir Himanshu"):
        response = f"Sir Himanshu, {response}"

    return response

# =========================================================
# DIRECT TINYLLAMA CHAT
# =========================================================

def ask_tinyllama(query):

    prompt = f"""
### System:
You are a respectful AI assistant.
You ALWAYS address the user as Sir Himanshu.

### Instruction:
{query}

### Response:
Sir Himanshu,
"""

    inputs = tiny_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        outputs = tiny_model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.65,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            eos_token_id=tiny_tokenizer.eos_token_id,
            pad_token_id=tiny_tokenizer.eos_token_id
        )

    response = tiny_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split("### Response:")[-1]

    stop_tokens = [
        "### Instruction:",
        "<|user|>",
        "<|assistant|>",
        "Q:",
        "User:"
    ]

    for token in stop_tokens:
        response = response.split(token)[0]

    response = response.strip()

    if not response.startswith("Sir Himanshu"):
        response = f"Sir Himanshu, {response}"

    return response

# =========================================================
# MAIN LOOP
# =========================================================

while True:

    query = input("\n🧑 You: ")

    if query.lower() == "exit":

        goodbye = "Goodbye Sir Himanshu. Have a wonderful day."

        print(f"\n🤖 Bot: {goodbye}")

        asyncio.run(
            speak_text(goodbye)
        )

        break

    # =====================================================
    # ROUTING
    # =====================================================

    if is_complex_query(query):

        print("\n🧠 Using Llama 3 reasoning...")

        smart_response = ask_llama3(query)

        print("✨ Applying TinyLlama personality...")

        final_response = rewrite_with_tinyllama(
            smart_response
        )

    else:

        print("\n⚡ Using TinyLlama...")

        final_response = ask_tinyllama(query)

    # =====================================================
    # OUTPUT
    # =====================================================

    print("\n🤖 Bot:", final_response)

    # =====================================================
    # SPEAK
    # =====================================================

    asyncio.run(
        speak_text(final_response)
    )


🔥 Using device: cuda
✅ TinyLlama loaded


tokenizer_config.json: 0.00B [00:00, ?B/s]

d:\transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\himan\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]